# Amazon Customer Data Preparation & Data Modeling

This project focuses on preparing and transforming Amazon customer and sales data into structured datasets ready for analysis and SQL-based data exploration.

Using **Python and Pandas**, the project loads the cleaned Amazon dataset, performs data preparation and transformation, and organizes the data into separate dimension and fact tables following a **star-schema approach**.

### Key Components

* Customer dimension (`dim_customer`)
* Product dimension (`dim_product`)
* Time dimension (`dim_time`)
* Sales fact table (`fact_sale`)
* Data type conversion and preparation
* Data transformation using Pandas
* Export of analytical tables to CSV files

The resulting datasets provide a clean and structured foundation for further **SQL analysis, Excel reporting, and business intelligence**.


In [ ]:
import pandas as pd

In [2]:
df = pd.read_csv("Amazon_cleaned.csv")

In [3]:
df_copy = df.copy()

In [4]:
df_copy.head(5)

,OrderID,OrderDate,OrderYear,OrderMonth,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,...,GrossAmount,DiscountedAmount,NetAmount,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
0,ORD0000001,2023-01-31,2023,January,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,...,319.77,0.00,319.77,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
1,ORD0000002,2023-12-30,2023,December,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,...,251.37,12.57,238.80,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
2,ORD0000003,2022-05-10,2022,May,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,...,105.09,10.51,94.58,108.06,Debit Card,Delivered,Austin,TX,United States,SELL00908
3,ORD0000004,2023-07-18,2023,July,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,...,167.90,25.18,142.72,159.66,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
4,ORD0000005,2023-02-04,2023,February,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,...,1031.28,257.82,773.46,821.36,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411


In [10]:
df_copy["OrderDate"]=pd.to_datetime(df_copy["OrderDate"])

### Create dim_customer

In [ ]:
dim_customer = (df_copy[["CustomerID","CustomerName","City","State","Country"]]
                .reset_index(drop=True)
                )
dim_customer

,CustomerID,CustomerName,City,State,Country
0,CUST001504,Vihaan Sharma,Washington,DC,India
1,CUST000178,Pooja Kumar,Fort Worth,TX,United States
2,CUST047516,Sneha Singh,Austin,TX,United States
3,CUST030059,Vihaan Reddy,Charlotte,NC,India
4,CUST048677,Aditya Kapoor,San Antonio,TX,Canada
...,...,...,...,...,...
99995,CUST001356,Karan Joshi,Jacksonville,FL,India
99996,CUST031254,Sunita Kapoor,San Jose,CA,United States
99997,CUST012579,Aman Gupta,Indianapolis,IN,United States
99998,CUST026243,Simran Gupta,Charlotte,NC,United States


### Create dim_products

In [ ]:
dim_product = (df_copy[["ProductID","ProductName","Category","Brand"]]
               .reset_index(drop=True)
               )
dim_product

,ProductID,ProductName,Category,Brand
0,P00014,Drone Mini,Books,BrightLux
1,P00040,Microphone,Home & Kitchen,UrbanStyle
2,P00044,Power Bank 20000mAh,Clothing,UrbanStyle
3,P00041,Webcam Full HD,Home & Kitchen,Zenith
4,P00029,T-Shirt,Clothing,KiddoFun
...,...,...,...,...
99995,P00047,Memory Card 128GB,Electronics,Apex
99996,P00046,Car Charger,Sports & Outdoors,Apex
99997,P00030,Dress Shirt,Sports & Outdoors,BrightLux
99998,P00046,Car Charger,Sports & Outdoors,HomeEase


### Create dim_time

In [14]:
dim_time = (df_copy[["OrderDate"]]
            .sort_values("OrderDate")
            .reset_index(drop=True)
            )
dim_time["DateKey"]=dim_time["OrderDate"].dt.strftime("%Y%m%d").astype(int)
dim_time["Year"] = dim_time["OrderDate"].dt.year
dim_time["Month"] = dim_time["OrderDate"].dt.month
dim_time["MonthName"] = dim_time["OrderDate"].dt.month_name()
dim_time["Quarter"] = dim_time["OrderDate"].dt.quarter
dim_time["Day"] = dim_time["OrderDate"].dt.day
dim_time

,OrderDate,DateKey,Year,Month,MonthName,Quarter,Day
0,2020-01-01,20200101,2020,1,January,1,1
1,2020-01-01,20200101,2020,1,January,1,1
2,2020-01-01,20200101,2020,1,January,1,1
3,2020-01-01,20200101,2020,1,January,1,1
4,2020-01-01,20200101,2020,1,January,1,1
...,...,...,...,...,...,...,...
99995,2024-12-29,20241229,2024,12,December,4,29
99996,2024-12-29,20241229,2024,12,December,4,29
99997,2024-12-29,20241229,2024,12,December,4,29
99998,2024-12-29,20241229,2024,12,December,4,29


In [15]:
df_copy.head(2)

,OrderID,OrderDate,OrderYear,OrderMonth,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,...,GrossAmount,DiscountedAmount,NetAmount,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
0,ORD0000001,2023-01-31,2023,January,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,...,319.77,0.00,319.77,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
1,ORD0000002,2023-12-30,2023,December,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,...,251.37,12.57,238.80,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298


### Create fact_sales

In [20]:
fact_sale = df_copy.merge(dim_time[["OrderDate","DateKey"]],
                          on="OrderDate",
                          how="left"
                          )
fact_sale = fact_sale[[
    "OrderID",
    "DateKey",
    "CustomerID",
    "ProductID",
    "SellerID",
    "Quantity",
    "UnitPrice",
    "Discount",
    "Tax",
    "ShippingCost",
    "GrossAmount",
    "DiscountedAmount",
    "NetAmount",
    "TotalAmount",
    "PaymentMethod",
    "OrderStatus"
]]
fact_sale

,OrderID,DateKey,CustomerID,ProductID,SellerID,Quantity,UnitPrice,Discount,Tax,ShippingCost,GrossAmount,DiscountedAmount,NetAmount,TotalAmount,PaymentMethod,OrderStatus
0,ORD0000001,20230131,CUST001504,P00014,SELL01967,3,106.59,0.00,0.00,0.09,319.77,0.00,319.77,319.86,Debit Card,Delivered
1,ORD0000001,20230131,CUST001504,P00014,SELL01967,3,106.59,0.00,0.00,0.09,319.77,0.00,319.77,319.86,Debit Card,Delivered
2,ORD0000001,20230131,CUST001504,P00014,SELL01967,3,106.59,0.00,0.00,0.09,319.77,0.00,319.77,319.86,Debit Card,Delivered
3,ORD0000001,20230131,CUST001504,P00014,SELL01967,3,106.59,0.00,0.00,0.09,319.77,0.00,319.77,319.86,Debit Card,Delivered
4,ORD0000001,20230131,CUST001504,P00014,SELL01967,3,106.59,0.00,0.00,0.09,319.77,0.00,319.77,319.86,Debit Card,Delivered
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5581757,ORD0100000,20211204,CUST029492,P00019,SELL00761,5,166.70,0.05,63.35,3.34,833.50,41.68,791.82,858.52,Debit Card,Delivered
5581758,ORD0100000,20211204,CUST029492,P00019,SELL00761,5,166.70,0.05,63.35,3.34,833.50,41.68,791.82,858.52,Debit Card,Delivered
5581759,ORD0100000,20211204,CUST029492,P00019,SELL00761,5,166.70,0.05,63.35,3.34,833.50,41.68,791.82,858.52,Debit Card,Delivered
5581760,ORD0100000,20211204,CUST029492,P00019,SELL00761,5,166.70,0.05,63.35,3.34,833.50,41.68,791.82,858.52,Debit Card,Delivered


### Export for SQL

In [21]:
dim_customer.to_csv("dim_customer.csv",index=False)
dim_product.to_csv("dim_product.csv",index=False)
dim_time.to_csv("dim_time.csv",index=False)
fact_sale.to_csv("fact_sale.csv",index=False)